In [3]:
import pandas as pd

# =========================================================================
# 1. ĐỌC DỮ LIỆU TỪ CÁC FILE SẠCH
# =========================================================================
print("Đang đọc các file dữ liệu...")
f1_partner = pd.read_csv('VNM_Seafood_Export_ByPartner_Cleaned_EN.csv')
f2_world = pd.read_csv('VNM_Seafood_World_Trade_With_Weight.csv')
f3_customs = pd.read_csv('VN_Seafood_2024_2025_Customs_VITIC.csv')

# Định nghĩa chuẩn cấu trúc và thứ tự cột để gộp
cols_order = ['Year', 'Trade_Flow', 'Partner_Country', 'HS_Code', 'Product_Description', 'Weight_KG', 'Value_USD']

# =========================================================================
# 2. CHUẨN HÓA CẤU TRÚC CHO FILE HẢI QUAN 2024 - 2025 (FILE 3)
# =========================================================================
f3_mapped = pd.DataFrame()
f3_mapped['Year'] = f3_customs['Year']
f3_mapped['Trade_Flow'] = f3_customs['Flow'].replace({'Exports': 'Export', 'Imports': 'Import'})
f3_mapped['Partner_Country'] = f3_customs['Partner']
f3_mapped['HS_Code'] = 'TOTAL'
f3_mapped['Product_Description'] = 'Total Seafood'
f3_mapped['Weight_KG'] = 0.0
f3_mapped['Value_USD'] = f3_customs['Trade_Value_USD']
f3_mapped = f3_mapped[cols_order]

# =========================================================================
# 3. TÁCH HÀNG VÀ LOẠI BỎ IMPORT TRONG FILE PARTNER (FILE 1)
# =========================================================================
print("Đang tiến hành tách hàng và loại bỏ các dòng Import ở file Partner...")

# Hàng 2 đến hàng 5 đưa vào File 2 (World) -> Giữ nguyên cả Export và Import (đủ 4 dòng)
f3_to_f2_world = f3_mapped.iloc[0:4].copy()

# Hàng 6 đến hàng 12 đưa vào File 1 (ByPartner) -> LỌC BỎ CHỈ GIỮ LẠI EXPORT
f3_to_f1_partner = f3_mapped.iloc[4:].copy()
# Lọc ở đây để loại bỏ Ấn Độ, Indonesia và Na Uy (vì 3 nước này là luồng Import)
f3_to_f1_partner = f3_to_f1_partner[f3_to_f1_partner['Trade_Flow'] == 'Export']

# =========================================================================
# 4. ĐỒNG BỘ VÀ TIẾN HÀNH GỘP (CONCATENATE)
# =========================================================================
if 'Weight_KG' not in f1_partner.columns:
    f1_partner['Weight_KG'] = 0.0

f1_partner_aligned = f1_partner[cols_order]
f2_world_aligned = f2_world[cols_order]

final_world_trade = pd.concat([f2_world_aligned, f3_to_f2_world], ignore_index=True)
final_partner_trade = pd.concat([f1_partner_aligned, f3_to_f1_partner], ignore_index=True)

# =========================================================================
# 5. XUẤT FILE KẾT QUẢ CUỐI CÙNG
# =========================================================================
output_world = 'Seafood_World_Trade_2015_2025_Final.csv'
output_partner = 'Seafood_ByPartner_Trade_2015_2025_Final.csv'

final_world_trade.to_csv(output_world, index=False, encoding='utf-8')
try:
    final_partner_trade.to_csv(output_partner, index=False, encoding='utf-8')
except PermissionError:
    alt_output_partner = output_partner.replace('.csv', '_new.csv')
    final_partner_trade.to_csv(alt_output_partner, index=False, encoding='utf-8')
    print(f"PermissionError: could not write '{output_partner}'. Saved to '{alt_output_partner}' instead. Close the file if it is open and rerun to overwrite the original.")

print("\n--- HOÀN THÀNH XỬ LÝ ---")
print(f"File 1 (Partner) mới tạo: '{output_partner}' -> Đã loại bỏ hoàn toàn các dòng Import.")
print(f"File 2 (World) mới tạo: '{output_world}' -> Giữ nguyên đầy đủ Xuất/Nhập khẩu.")

Đang đọc các file dữ liệu...
Đang tiến hành tách hàng và loại bỏ các dòng Import ở file Partner...
PermissionError: could not write 'Seafood_ByPartner_Trade_2015_2025_Final.csv'. Saved to 'Seafood_ByPartner_Trade_2015_2025_Final_new.csv' instead. Close the file if it is open and rerun to overwrite the original.

--- HOÀN THÀNH XỬ LÝ ---
File 1 (Partner) mới tạo: 'Seafood_ByPartner_Trade_2015_2025_Final.csv' -> Đã loại bỏ hoàn toàn các dòng Import.
File 2 (World) mới tạo: 'Seafood_World_Trade_2015_2025_Final.csv' -> Giữ nguyên đầy đủ Xuất/Nhập khẩu.
